# Lesson 14 Lab — Weight Quantization Deployment Contracts

**Puzzle:** Why can an AWQ, GPTQ, or FP8 checkpoint fail even when vLLM supports that method?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A quantization label is only one field of a deployment contract. GPU capability, weight layout, group size, activation dtype, model architecture, loader metadata, and kernel availability must agree.


## 0. Predict before running

1. Identify the local checkpoint's declared quantization config.
2. Probe whether AWQ, GPTQ, and FP8 names are registered.
3. State why no latency comparison is made in this lesson.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The compatibility lab inspects vLLM's registered quantization methods and engine CLI, reads the unquantized local checkpoint metadata, and evaluates a declared RTX 5090 compatibility matrix without downloading substitute models.

- Loader recognition is weaker than kernel dispatch.
- Hardware support does not validate checkpoint metadata.
- Memory, quality, and latency require separate gates.


## 2. Derive the mechanism

Weight-only AWQ and GPTQ store codes plus scale metadata and depend on kernels that understand their packing. FP8 may target weights and/or activations with hardware-specific execution. A loader can recognize a format yet fall back, reject an architecture, or execute with no speedup at the tested shape. Native model artifacts are required for performance evidence.

### Mechanism at a glance

```mermaid
flowchart TD
  C["quantized checkpoint metadata"] --> L{"loader supports format?"}
  H["GPU capability"] --> K{"native kernel available?"}
  L --> K
  K --> E["execute frozen workload"]
  E --> G{"quality + memory + latency gates"}
  G -->|"pass"| P["promote route"]
  G -->|"fail"| R["rollback to BF16"]
```

### Walk it step by step

1. **Inspect the checkpoint.** Read format, group, scale, and architecture metadata.
2. **Match the platform.** Verify the GPU and compiled kernel prerequisites.
3. **Prove dispatch.** Use native logs or traces, not a config label.
4. **Gate the product result.** Evaluate quality, memory, and service latency separately.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 14
LESSON_TITLE = 'Weight Quantization Deployment Contracts'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260826
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | local BF16 checkpoint |
| Candidate | AWQ, GPTQ, and FP8 candidate contracts |
| Held constant | vLLM build, GPU, local config, and no network download |
| Measurements | registered methods, checkpoint declaration, hardware capability, readiness fields, and missing evidence |
| Evidence | `compatibility-probe` |

**Experiment:** Probe installed quantization registrations and evaluate checkpoint/hardware prerequisites for three deployment routes.


## 5. Inspect the experiment code

The notebook imports registries defensively because internal module paths can change. A failed probe is retained as compatibility evidence rather than converted into a success claim.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); registered=[]; probe_error=None
try:
    module=importlib.import_module("vllm.model_executor.layers.quantization")
    registered=sorted(str(x).lower() for x in getattr(module,"QUANTIZATION_METHODS",[]))
except Exception as exc: probe_error=f"{type(exc).__name__}: {exc}"
_,serve_help=cli_help("serve"); blob=" ".join(registered)+" "+serve_help.lower(); declared=cfg.get("quantization_config")
metrics={"local":{"quantization":"none" if declared is None else str(declared),"torch_dtype":str(cfg.get("torch_dtype"))},
         "methods":{"awq":"awq" in blob,"gptq":"gptq" in blob,
                    "fp8":"fp8" in blob or "compressed-tensors" in blob},
         "hardware":{"compute_capability":ENV["compute_capability"],"gpu":ENV["gpu"]},
         "registered_methods":registered,"probe_error":probe_error,"native_quantized_benchmark_completed":False}
analysis=(f"Local quantization={metrics['local']['quantization']}; installed AWQ/GPTQ/FP8 vocabulary="
          f"{metrics['methods']['awq']}/{metrics['methods']['gptq']}/{metrics['methods']['fp8']}. "
          "Without matching quantized bytes, memory, quality, and latency remain unmeasured.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Declared local quantization | none |
| AWQ registered | yes |
| GPTQ registered | yes |
| FP8 registered | yes |
| Compute capability | 12.0 |
| Native quantized benchmark | not measured |


## 7. Explain the result

Local quantization=none; installed AWQ/GPTQ/FP8 vocabulary=True/True/True. Without matching quantized bytes, memory, quality, and latency remain unmeasured.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 14, "title": 'Weight Quantization Deployment Contracts', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'This probe maps available software vocabulary and missing prerequisites; it deliberately makes no quantized-performance claim.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 14,
  "title": "Weight Quantization Deployment Contracts",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260826
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "local": {
      "quantization": "none",
      "torch_dtype": "bfloat16"
    },
    "methods": {
      "awq": true,
      "gptq": true,
      "fp8": true
    },
    "hardware": {
      "compute_capability": "12.0",
      "gpu": "NVIDIA GeForce RTX 5090"
    },
    "registered_methods": [
      "auto_awq",
      "auto_gptq",
      "awq",
      "awq_marlin",
      "bitsandbytes",
      "compressed-tensors",
      "deepseek_v4_fp8",
      "experts_int8",
      "fbgemm_fp8",
      "fp8",
      "fp8_per_block",
      "fp8_per_channel",
      "fp8_per_tensor",
      "fp_quant",
      "gpt_oss_mxfp4",


## 9. Make the bounded decision

> This probe maps available software vocabulary and missing prerequisites; it deliberately makes no quantized-performance claim.

**Acceptance/rollback:** Benchmark a quantized route only after its exact checkpoint loads, a native trace identifies the intended path, and output quality passes.

**Failure analysis:** Registry presence can outlive a deprecated path or omit platform-specific constraints. This lab has no AWQ/GPTQ/FP8 weight artifact and therefore cannot measure their memory or speed.


## 10. Extend the evidence

Pin one quantized checkpoint per route, hash it, run the same prompt grid, capture kernel traces, and compare quality plus memory against BF16.

The full boundary and references are in [`README.md`](README.md).
